In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")

# 1) Load all core CSVs (men + women)
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
w_teams = pd.read_csv(DATA_DIR / "WTeams.csv")
m_regular = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
w_regular = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")
sample_sub = pd.read_csv(DATA_DIR / "SampleSubmissionStage1.csv")

print("Data loaded")
print(f"Men's teams:   {len(m_teams):5d}")
print(f"Women's teams: {len(w_teams):5d}")
print(f"Men regular:   {len(m_regular):5d} games")
print(f"Women regular: {len(w_regular):5d} games")
print(f"Men tourney:   {len(m_tourney):5d} games")
print(f"Women tourney: {len(w_tourney):5d} games")
print(f"Sample submission rows: {len(sample_sub):d}")

all_reg = pd.concat([m_regular.assign(Gender="M"),
                     w_regular.assign(Gender="W")], ignore_index=True)
all_tour = pd.concat([m_tourney.assign(Gender="M"),
                      w_tourney.assign(Gender="W")], ignore_index=True)

print("\nSeasons (men regular):", m_regular["Season"].min(), "-", m_regular["Season"].max())
print("Total regular season games (M + W):", len(all_reg))
print("Total tourney games (M + W):", len(all_tour))

Data loaded
Men's teams:     381
Women's teams:   379
Men regular:   196823 games
Women regular: 140825 games
Men tourney:    2585 games
Women tourney:  1717 games
Sample submission rows: 519144

Seasons (men regular): 1985 - 2026
Total regular season games (M + W): 337648
Total tourney games (M + W): 4302


In [16]:
# Basic EDA
print("\nRegular season columns:", list(all_reg.columns))
print("Regular season shape:", all_reg.shape)
print("Tourney shape:", all_tour.shape)

print("\nExample regular-season rows (men):")
print(m_regular.head())

print("\nExample regular-season rows (women):")
print(w_regular.head())

print("\nTeam ID ranges:")
print("Men:", m_teams["TeamID"].min(), "-", m_teams["TeamID"].max())
print("Women:", w_teams["TeamID"].min(), "-", w_teams["TeamID"].max())

print("\nSample submission preview:")
print(sample_sub.head())
print("Distinct seasons in sample_sub:", sample_sub["ID"].str[:4].unique())


Regular season columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'Gender']
Regular season shape: (337648, 9)
Tourney shape: (4302, 9)

Example regular-season rows (men):
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985      20     1228      81     1328      64    N      0
1    1985      25     1106      77     1354      70    H      0
2    1985      25     1112      63     1223      56    H      0
3    1985      25     1165      70     1432      54    H      0
4    1985      25     1192      86     1447      74    H      0

Example regular-season rows (women):
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1998      18     3104      91     3202      41    H      0
1    1998      18     3163      87     3221      76    H      0
2    1998      18     3222      66     3261      59    H      0
3    1998      18     3307      69     3365      62    H      0
4    1998      18     3349     115     3411     

In [3]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("../src")))

from features import (
    normalize_games,
    build_team_season_stats,
    build_matchup_dataset,
)
print("Importing complete!")

Importing complete!


In [4]:
m_reg = pd.read_csv("../data/raw/MRegularSeasonCompactResults.csv")
w_reg = pd.read_csv("../data/raw/WRegularSeasonCompactResults.csv")
print("Data read!")

Data read!


In [5]:
m_norm = normalize_games(m_reg, "M")
w_norm = normalize_games(w_reg, "W")

games = pd.concat([m_norm, w_norm])

print("Total games:", len(games))
print("Missing values:\n", games.isna().sum())
print("Unique seasons:", games["Season"].nunique())
print("Team1Win distribution:")
print(games["Team1Win"].value_counts(normalize=True))

Total games: 337648
Missing values:
 Season       0
DayNum       0
Team1        0
Team2        0
Team1Win     0
PointDiff    0
Gender       0
dtype: int64
Unique seasons: 42
Team1Win distribution:
Team1Win
0    0.505592
1    0.494408
Name: proportion, dtype: float64


In [6]:
team_stats = build_team_season_stats(games)

print("Team stats shape:", team_stats.shape)
print(team_stats.head())
print("WinPct range:",
      team_stats["WinPct"].min(),
      team_stats["WinPct"].max())

Team stats shape: (23604, 6)
   Season  TeamID  Games  Wins  AvgPointDiff    WinPct
0    1985    1102     24     5     -5.791667  0.208333
1    1985    1103     23     9     -3.043478  0.391304
2    1985    1104     30    21      7.800000  0.700000
3    1985    1106     24    10     -3.791667  0.416667
4    1985    1108     25    19     11.173913  0.760000
WinPct range: 0.0 1.0


In [7]:
matchups = build_matchup_dataset(games, team_stats)

print("Matchups shape:", matchups.shape)
print(matchups.head())
print(matchups.isna().sum())

Matchups shape: (337648, 6)
   Season  Team1  Team2  WinPctDiff  AvgPDDiff  Team1Win
0    1985   1228   1328   -0.091398   4.158654         1
1    1985   1106   1354    0.041667  11.799242         1
2    1985   1112   1223   -0.013333   2.553775         1
3    1985   1165   1432    0.021739  -6.257937         1
4    1985   1192   1447    0.345238  11.433333         1
Season        0
Team1         0
Team2         0
WinPctDiff    0
AvgPDDiff     0
Team1Win      0
dtype: int64


In [8]:
print("WinPctDiff correlation:",
      matchups["WinPctDiff"].corr(matchups["Team1Win"]))

print("AvgPDDiff correlation:",
      matchups["AvgPDDiff"].corr(matchups["Team1Win"]))

WinPctDiff correlation: 0.566876111768954
AvgPDDiff correlation: 0.5133976289537132


In [9]:
DATA_PATH = Path("../data/raw")

m_reg = pd.read_csv(DATA_PATH / "MRegularSeasonCompactResults.csv")
w_reg = pd.read_csv(DATA_PATH / "WRegularSeasonCompactResults.csv")
m_tour = pd.read_csv(DATA_PATH / "MNCAATourneyCompactResults.csv")
w_tour = pd.read_csv(DATA_PATH / "WNCAATourneyCompactResults.csv")

print("Men Regular:", m_reg.shape)
print("Women Regular:", w_reg.shape)
print("Men Tourney:", m_tour.shape)
print("Women Tourney:", w_tour.shape)

Men Regular: (196823, 8)
Women Regular: (140825, 8)
Men Tourney: (2585, 8)
Women Tourney: (1717, 8)


In [10]:
m_reg_norm = normalize_games(m_reg, "M")
w_reg_norm = normalize_games(w_reg, "W")
m_tour_norm = normalize_games(m_tour, "M")
w_tour_norm = normalize_games(w_tour, "W")

all_games = pd.concat(
    [m_reg_norm, w_reg_norm, m_tour_norm, w_tour_norm],
    ignore_index=True
)

print("Total games:", len(all_games))
all_games.head()

Total games: 341950


,Season,DayNum,Team1,Team2,Team1Win,PointDiff,Gender
0,1985,20,1228,1328,1,17,M
1,1985,25,1106,1354,1,7,M
2,1985,25,1112,1223,1,7,M
3,1985,25,1165,1432,1,16,M
4,1985,25,1192,1447,1,12,M


In [11]:
print("Team1 win rate:", all_games["Team1Win"].mean())
print("PointDiff summary:")
print(all_games["PointDiff"].describe())

Team1 win rate: 0.49465418920894866
PointDiff summary:
count    341950.000000
mean         -0.181249
std          16.605840
min         -98.000000
25%         -11.000000
50%          -1.000000
75%          11.000000
max         108.000000
Name: PointDiff, dtype: float64


In [12]:
team_stats = build_team_season_stats(all_games)

print("Total games:", len(all_games))
print("Total team-seasons:", len(team_stats))
team_stats.head()

Total games: 341950
Total team-seasons: 23604


,Season,TeamID,Games,Wins,AvgPointDiff,WinPct
0,1985,1102,24,5,-5.791667,0.208333
1,1985,1103,23,9,-3.043478,0.391304
2,1985,1104,33,23,7.303030,0.696970
3,1985,1106,24,10,-3.791667,0.416667
4,1985,1108,25,19,11.173913,0.760000


In [13]:
print(team_stats["WinPct"].describe())
print(team_stats["AvgPointDiff"].describe())

count    23604.000000
mean         0.490727
std          0.193397
min          0.000000
25%          0.354839
50%          0.500000
75%          0.633333
max          1.000000
Name: WinPct, dtype: float64
count    23604.000000
mean        -0.279184
std          8.192189
min        -49.777778
25%         -5.506474
50%         -0.172318
75%          4.966667
max         49.767857
Name: AvgPointDiff, dtype: float64


In [14]:
matchups = build_matchup_dataset(all_games, team_stats)

print("Matchup rows:", len(matchups))
matchups.head()

Matchup rows: 341950


,Season,Team1,Team2,WinPctDiff,AvgPDDiff,Team1Win
0,1985,1228,1328,-0.088235,3.280423,1
1,1985,1106,1354,0.041667,11.799242,1
2,1985,1112,1223,-0.037143,-5.538818,1
3,1985,1165,1432,0.021739,-6.257937,1
4,1985,1192,1447,0.321839,11.314103,1


In [15]:
print("WinPctDiff correlation:",
      matchups["WinPctDiff"].corr(matchups["Team1Win"]))

print("AvgPDDiff correlation:",
      matchups["AvgPDDiff"].corr(matchups["Team1Win"]))

WinPctDiff correlation: 0.5646668346047552
AvgPDDiff correlation: 0.5100809550122478
